In [2]:
import numpy as np
import random

# =========================
# ЧТЕНИЕ ДАННЫХ
# =========================
# N — número de datos (cantidad de puntos)
# K — número de atributos (dimensión de cada punto)
# M — número de clusters (grupos)

# N — количество объектов (размер набора данных)
# K — размерность данных (число признаков у каждой точки)
# M — число кластеров, на которые делим данные
from sklearn.datasets import make_blobs

def leer_datos():
    # генерируем синтетические данные (кластеризованные)
    X, y_true = make_blobs(
        n_samples=100,   # число точек (N)
        centers=3,       # число кластеров (M)
        n_features=2,    # размерность (K)
        cluster_std=0.3, # разброс (малый → легко кластеризуется)
        random_state=42
    )
    
    # N — число объектов
    N = X.shape[0]
    
    # K — число признаков
    K = X.shape[1]
    
    # M — число кластеров
    M = 3
    
    return np.array(X), N, K, M, y_true

In [3]:

# =========================
# ВЫЧИСЛЕНИЕ ЦЕНТРОВ
# =========================
# вычисляет центры кластеров на основе разбиения P и данных X
def calcular_medoides(P, X, M):
    """
    P - разбиение (вектор кластеров)
    X - данные
    M - число кластеров
    
    Возвращает medoids (представители кластеров из X)
    """
    
    K = X.shape[1]
    medoids = np.zeros((M, K))
    
    for j in range(M):
        # выбираем точки кластера j
        puntos = X[P == j]
        
        if len(puntos) > 0:
            # считаем сумму расстояний для каждой точки
            mejor_punto = None
            mejor_dist = float('inf')
            
            for p in puntos:
                suma = 0.0
                
                for q in puntos:
                    suma += np.linalg.norm(p - q)
                
                # ищем точку с минимальной суммой расстояний
                if suma < mejor_dist:
                    mejor_dist = suma
                    mejor_punto = p
            
            medoids[j] = mejor_punto
        
        else:
            # если кластер пуст — случайная точка
            medoids[j] = X[random.randint(0, len(X)-1)]
    
    return medoids


In [4]:

# =========================
# FITNESS (ЦЕЛЕВАЯ ФУНКЦИЯ)
# =========================

def fitness(P, X, M):
    # получаем центр каждого кластера
    medoids = calcular_medoides(P, X, M)
    
    total = 0.0
    # считаем сумму расстояний` от каждой точки до центра её кластера`
    for i in range(len(X)):
        c = medoids[P[i]]
        total += np.linalg.norm(X[i] - c)**2
    
    return total

In [5]:

# =========================
# ИНИЦИАЛИЗАЦИЯ ПОПУЛЯЦИИ
# =========================
def inicializar_poblacion(popSize, N, M):
    poblacion = []
    # Создаем количество возможных решений, равное размеру популяции
    for _ in range(popSize):
        # случайное разбиение, где M — число кластеров, а N — количество точек
        individuo = np.random.randint(0, M, size=N)
        # Добавляем созданное решение в популяцию
        poblacion.append(individuo)
    
    return np.array(poblacion)


In [6]:
# =========================
# ОЦЕНКА ПОПУЛЯЦИИ
# =========================

def evaluar_poblacion(poblacion, X, M):
    fitness_vals = []
    # оцениваем каждое разбиение в популяции
    for ind in poblacion:
        fitness_vals.append(fitness(ind, X, M))
    
    return np.array(fitness_vals)

In [7]:
# =========================
# ОБУЧЕНИЕ РАСПРЕДЕЛЕНИЯ
# =========================

def aprender_distribucion(mejores, N, M):
    """
    mejores: лучшие решения
    возвращает P(i,j) — вероятность, что элемент i в кластере j
    """
    # N - количество точек
    # M - число кластеров
    distribucion = np.zeros((N, M))
    # Создаём матрицу вероятностей, где distribucion[i, j] — это вероятность того, что точка i принадлежит кластеру j. 
    # Изначально она заполнена нулями.
    for i in range(N):
    # в каком кластере чаще всего находится точка i
        for ind in mejores:
            distribucion[i, ind[i]] += 1
        
        distribucion[i] /= len(mejores)
    
    return distribucion

In [8]:
# =========================
# ГЕНЕРАЦИЯ НОВОЙ ПОПУЛЯЦИИ
# =========================

def muestrear(distribucion, popSize, N, M):
    # distribucion — матрица вероятностей размера (N x M)
    # distribucion[i][j] = вероятность того, что точка i попадёт в кластер j

    nueva_poblacion = []  # сюда будем сохранять новые решения

    # создаём popSize новых решений (индивидов)
    for _ in range(popSize):
        
        # создаём пустой индивид (разбиение)
        # длина N — по одному кластеру для каждой точки
        individuo = np.zeros(N, dtype=int)
        
        # для каждой точки i выбираем кластер
        for i in range(N):
            
            # np.random.choice выбирает случайное значение из {0,...,M-1}
            # но НЕ равновероятно, а по заданным вероятностям distribucion[i]
            
            # distribucion[i] — это, например:
            # [0.7, 0.2, 0.1]
            # значит:
            # 70% → кластер 0
            # 20% → кластер 1
            # 10% → кластер 2
            
            individuo[i] = np.random.choice(range(M), p=distribucion[i])
        
        # добавляем готовое решение в популяцию
        nueva_poblacion.append(individuo)
    
    # преобразуем список в numpy-массив
    return np.array(nueva_poblacion)

In [9]:
# =========================
# ОСНОВНОЙ АЛГОРИТМ EDA
# =========================
# 🔹 fichero 👉 путь к файлу с данными
# содержит точки (dataset)
# пример: "test.txt"
# 🔹 popSize
# 👉 размер популяции
# сколько решений (разбиений) храним на каждой итерации
# больше → лучше поиск, но медленнее
# 🔹 numPadres
# 👉 число лучших решений для обучения
# сколько "лучших" используем, чтобы построить распределение
# меньше → сильнее давление отбора
# больше → больше разнообразия
# 🔹 iteraciones
# 👉 число итераций алгоритма
# сколько раз повторяем цикл EDA
# больше → лучше результат, но дольше
def EDA_clustering(fichero, popSize=50, numPadres=10, iteraciones=50):
    
    # читаем данные
    X, N, K, M, y_true = leer_datos()
    
    # начальная популяция
    poblacion = inicializar_poblacion(popSize, N, M)
    # лучшее разбиение (индивид)
    mejor_global = None
    # лучшее значение лучшего разбиения
    # float('inf') — начальное значение, так как задача — минимизация:
    # мы ищем разбиение, минимизирующее сумму квадратов расстояний от точек до центров кластеров
    mejor_fitness = float('inf')
    
    for it in range(iteraciones):
        
        # оцениваем
        # ппосчитаем fitness для каждого решения
        fitness_vals = evaluar_poblacion(poblacion, X, M)
        
        # сортируем
        orden = np.argsort(fitness_vals)
        poblacion = poblacion[orden]
        # после сортировки fitness_vals[0] — это лучший
        # fitness в текущей популяции
        # fitness_vals[-1] → худший
        fitness_vals = fitness_vals[orden]
        # мы минимизируем fitness
        # меньше = лучше
        # argsort сортирует по возрастанию
        
        # обновляем лучшее
        if fitness_vals[0] < mejor_fitness:
            mejor_fitness = fitness_vals[0]
            mejor_global = poblacion[0].copy()
        
        # выбираем лучших
        mejores = poblacion[:numPadres]
        
        # учим распределение
        distribucion = aprender_distribucion(mejores, N, M)
        
        # генерируем новую популяцию
        poblacion = muestrear(distribucion, popSize, N, M)
         
        print(f"Iter {it}: mejor = {fitness_vals[0]:.4f}, media = {np.mean(fitness_vals):.4f}")
    
    print("\n=== RESULTADO FINAL ===")
    print("Mejor fitness:", mejor_fitness)
    print("Mejor partición:", mejor_global)
    return mejor_fitness, mejor_global, y_true

In [12]:
# =========================
# ЗАПУСК
# =========================

mejor_fitness, mejor_global, y_true= EDA_clustering("test.txt", 100, 20, 50)

Iter 0: mejor = 9083.4702, media = 9936.9643
Iter 1: mejor = 7932.9760, media = 9856.8796
Iter 2: mejor = 8488.3389, media = 9889.1961
Iter 3: mejor = 8270.0127, media = 9847.5865
Iter 4: mejor = 8025.6019, media = 9494.8048
Iter 5: mejor = 6764.4782, media = 8762.8192
Iter 6: mejor = 5272.8992, media = 7817.4726
Iter 7: mejor = 5183.1112, media = 6712.1640
Iter 8: mejor = 3519.4524, media = 5691.1310
Iter 9: mejor = 2642.0609, media = 4557.6428
Iter 10: mejor = 2434.8094, media = 3646.3732
Iter 11: mejor = 2114.7685, media = 2870.0895
Iter 12: mejor = 1501.9502, media = 2323.7351
Iter 13: mejor = 991.8523, media = 1851.1160
Iter 14: mejor = 703.3084, media = 1411.5034
Iter 15: mejor = 688.7835, media = 1067.8518
Iter 16: mejor = 511.8223, media = 777.4631
Iter 17: mejor = 511.8223, media = 602.7238
Iter 18: mejor = 511.8223, media = 511.8223
Iter 19: mejor = 511.8223, media = 511.8223
Iter 20: mejor = 511.8223, media = 511.8223
Iter 21: mejor = 511.8223, media = 511.8223
Iter 22: mejo

In [13]:
from sklearn.metrics import adjusted_rand_score

print(adjusted_rand_score(y_true, mejor_global))

0.8576388251070005
